## Creating vp_data3

Creating database *vp_data3.db* that consists of verb rection patterns and various index tables for transaction database matches. The resulting database tables are described in [*table_descriptions*](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/004_analysis_of_known_verb_rection_patterns/table_descriptions).

*vp_data3.db* is based on only one-member patterns from *verb_patterns_new.db* that contain a verb with up to 1 compound part and do not contain a word belonging into *other* category (this category is difficult to solve). Also, the patterns are chosen based on their appearance in transactions database (only ones containing a verb that occurs in transactions are chosen).

In [1]:
import sys

sys.path.append('../../../common_code')

In [2]:
import sqlite3
from db_operations.db_display import *

## Input parameters

In [3]:
INPUT_DIR = "C:/Users/liivas/Documents/Töö/verbirektisoonid"
DB_DIR = "../001_creating_pattern_tables"

RESULT_DB = "vp_data3.db"
TRANSACTION_DB = f"{INPUT_DIR}/v32_data.db"
VERB_PATTERNS_DB = f"{DB_DIR}/verb_patterns_new.db"

## Data processing

In [ ]:
con = sqlite3.connect(VERB_PATTERNS_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS v32')
cur.execute(f'ATTACH DATABASE "{RESULT_DB}" AS vp')

### I table patterns

Veerud:

    pat_id - (mustri ID tabelis patterns_len1)
    pattern - (algne muster sõnena)
    verb_word - (mustri (pea)verb)
    verb_compound - (pikema verbiühendi ülejäänud osad)
    phrase_nr - (fraasi number; kuna hetkel on vaatluse all ainult tabelist patterns_len1 pärit mustrid, on kõigil fraasidel number 1)
    phrase_case - (fraasi põhiliikme (pärast verbi) kääne; hiljem vaatame ilmselt vaid fraase, kus selles käändes on obliikva, kuid praeguseks pole seda tingimust veel sisse pandud)
    adp - (kaassõna)
    inf_verb - (infiniitverb)
    
Vajalik info saadakse tabelitest patterns_len1 ja transaction_head. Tabelis *patterns* on tabelist patterns_len1 mustrid, millel on lisaks põhiverbile kuni üks verbiühendi osa ning millel ei ole tabelis patterns_len1 *other*-kategooriasse kuuluvat mustriliiget.
Verbiühendeid, millel on lisaks põhiverbile rohkem, kui üks osa, on transaktsioonide andmebaasis võrdlemisi vähe.
*Other*-kategooria lahendamine on natuke keerukas (sidesõnad, sõnad nagu 'millal', 'kuidas' jne).

In [5]:
cur.execute("""
DROP TABLE IF EXISTS vp.patterns
""")

cur.execute("""
CREATE TABLE vp.patterns AS
SELECT DISTINCT
    pat.ID AS pat_id,
    pat.word || ' ' || pat.government AS pattern,
    pat.verb_word AS verb_word,
    pat.compound_prt1 AS verb_compound,
    pat.phrase_nr AS phrase_nr,
    pat.w_case AS phrase_case,
    pat.adp AS adp,
    pat.verb AS inf_verb
FROM 
    patterns_len1 as pat
INNER JOIN 
    v32.transaction_head
ON 
    pat.verb_word = v32.transaction_head.verb
WHERE 
    pat.compound_prt1 = v32.transaction_head.verb_compound
AND 
    pat.compound_prt2 = ''
AND 
    pat.compound_prt3 = ''
AND 
    pat.other = ''
""")

cur.execute("""
CREATE INDEX vp.pat_id_idx ON patterns(pat_id)
"""
)

cur.execute("""
CREATE INDEX vp.phrase_case_idx ON patterns(phrase_case)
"""
)

cur.execute("""
CREATE INDEX vp.adp_idx ON patterns(adp)
"""
)

cur.execute("""
CREATE INDEX vp.inf_verb_idx ON patterns(inf_verb)
"""
)

cur.execute("""
CREATE INDEX vp.verb_word_idx ON patterns(verb_word)
"""
)

cur.execute("""
CREATE INDEX vp.verb_compound_idx ON patterns(verb_compound)
"""
)

cur.execute("""
CREATE INDEX vp.phrase_nr_idx ON patterns(phrase_nr)
"""
)

### Helper table

Mitte-elegantne viis saada kätte kõik **head_id**-d, millele vastavates fraasides on esindatud kõik vaadeldavate mustrite osised (sobiv kääne (kui on), kaassõna (kui on), infiniitverb (kui on)). Saab kasutada ülejäänud tabelite koostamiseks.

In [6]:
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step1
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step2
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step3
""")

cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step1 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
(
    SELECT pat.pat_id as pat_id,
        tr_head.id as head_id,
        pat.phrase_case as phrase_case,
        pat.adp as adp,
        pat.inf_verb as inf_verb,
        pat.phrase_nr as phrase_nr
    FROM 
        vp.patterns as pat
    INNER JOIN 
        v32.transaction_head as tr_head
    ON
        pat.verb_word=tr_head.verb
    WHERE
        pat.verb_compound=tr_head.verb_compound
) as pat_tr_joined
INNER JOIN 
    v32.`transaction_row` as tr
ON
    pat_tr_joined.head_id=tr.head_id
WHERE 
    pat_tr_joined.phrase_case = '' OR instr(tr.feats, pat_tr_joined.phrase_case) > 0
""")


cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step2 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
    vp.verb_phrase_matches_step1 as step1
INNER JOIN 
    v32.`transaction_row` as tr
ON 
    step1.head_id=tr.head_id
WHERE 
    step1.adp = '' OR (tr.form = step1.adp AND tr.deprel = 'case')
""")


cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step3 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb, 
    phrase_nr
FROM 
    vp.verb_phrase_matches_step2 as step2
INNER JOIN
    v32.`transaction_row` as tr
ON 
    step2.head_id=tr.head_id
WHERE 
    step2.inf_verb = '' OR (tr.form = step2.inf_verb AND (instr(tr.feats, 'inf') > 0 OR instr(tr.feats, 'sup') > 0))
""")


cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step1
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step2
""")

### II table patterns_meta

Veerud:

    pat_id - mustri ID
    phrase_count - mustrile vastavate fraaside (limiteeritud) hulk transaktsioonide andmebaasis
    
Asjade kättesaamiseks kasutatakse abitabelit, sealt leitakse esiteks unikaalsed read, mis seejärel kokku loetakse.

In [7]:
cur.execute("""
DROP TABLE IF EXISTS vp.patterns_meta
""")

cur.execute("""
CREATE TABLE vp.patterns_meta (
    pat_id INTEGER,
    phrase_count INTEGER
)
""")

cur.execute("""
INSERT INTO vp.patterns_meta(
    pat_id,
    phrase_count
)
SELECT 
    pat_id,
    count(*) AS phrase_count
FROM
(
    SELECT DISTINCT
        pat_id, 
        head_id
    FROM
        vp.verb_phrase_matches_step3
) AS tbl
GROUP BY
    tbl.pat_id
ORDER BY
    phrase_count DESC
""")

cur.execute("""
CREATE INDEX vp.meta_pat_id_idx ON patterns_meta(pat_id)
"""
)

con.commit()

### III table verb_phrase_matches

Veerud:

    pat_id - mustri ID
    head_id - verbi ID transaktsioonide andmebaasis
    phrase_nr - fraasi nr
    
Asjade kättesaamiseks kasutatakse abitabelit, sealt leitakse unikaalsed read.
Kui tulevikus võtta juurde pikemad mustrid, mis sisaldavad rohkem, kui ühte fraasi, siis saab esimene fraas olema eristatud tähistatud numbriga 1 ning teine numbriga 2.

In [8]:
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches
""")

cur.execute("""
CREATE TABLE vp.verb_phrase_matches (
    pat_id INTEGER,
    head_id INTEGER,
    phrase_nr INTEGER
)
""")

cur.execute("""
INSERT INTO vp.verb_phrase_matches(
    pat_id,
    head_id,
    phrase_nr
)
SELECT DISTINCT
    pat_id,
    head_id,
    phrase_nr
FROM
    vp.verb_phrase_matches_step3
""")

cur.execute("""
CREATE INDEX vp.match_pat_id_idx ON verb_phrase_matches(pat_id)
"""
)

cur.execute("""
CREATE INDEX vp.match_head_id_idx ON verb_phrase_matches(head_id)
"""
)

cur.execute("""
CREATE INDEX vp.match_phrase_nr_idx ON verb_phrase_matches(phrase_nr)
"""
)


con.commit()

### IV table verb_matches

Veerud:

    pat_id - mustri ID
    head_id - verbi ID transaktsioonide andmebaasis
    
Asjade kättesaamiseks kasutatakse tabelit patterns, et saada kätte mustrite verbid ning transaktsioonide tabelit transaction_head, et saada kätte viited kõigile lausetele, kus verbid esinevad. Ülejäänud mustrit pole selle tabeli puhul arvesse võetud.

In [9]:
# tabel verb_matches
cur.execute("""
DROP TABLE IF EXISTS vp.verb_matches
""")

cur.execute("""
CREATE TABLE vp.verb_matches (
    pat_id INTEGER,
    head_id INTEGER
)
""")

cur.execute("""
INSERT INTO vp.verb_matches(
    pat_id,
    head_id
)
SELECT DISTINCT
    pat_id,
    head_id
FROM
(
    SELECT pat_id, tr_head.id as head_id
    FROM
        patterns as pat
    INNER JOIN
        v32.transaction_head as tr_head
    ON
        pat.verb_word = tr_head.verb
    WHERE pat.verb_compound = tr_head.verb_compound
)
""")

cur.execute("""
CREATE INDEX vp.v_match_pat_id_idx ON verb_matches(pat_id)
"""
)

cur.execute("""
CREATE INDEX vp.v_match_head_id_idx ON verb_matches(head_id)
"""
)


con.commit()

In [ ]:
# soovi korral saab kustutada abitabeli, aga SQLITE-s see tegevus mäluruumi ei vabasta

#cur.execute("""
#DROP TABLE IF EXISTS vp.verb_phrase_matches_step3
#""")

In [10]:
# ühenduse sulgemine
con.close()

## Result

In [5]:
display_sqlite_as_dataframe(RESULT_DB, 'patterns', 10)

,pat_id,pattern,verb_word,verb_compound,phrase_nr,phrase_case,adp,inf_verb
0,1,aasima keda*,aasima,,1,part,,
1,2,aasima kelle kallal,aasima,,1,gen,kallal,
2,3,abielluma kellega,abielluma,,1,kom,,
3,5,abstraheeruma millest/kellest,abstraheeruma,,1,el,,
4,6,adresseerima mida*,adresseerima,,1,part,,
5,7,adresseerima kellele,adresseerima,,1,all,,
6,8,aevastama mille peale,aevastama,,1,gen,peale,
7,9,agiteerima keda*,agiteerima,,1,part,,
8,10,ahistama keda*,ahistama,,1,part,,
9,11,ahvima keda*,ahvima,,1,part,,


In [6]:
display_sqlite_as_dataframe(RESULT_DB, 'verb_phrase_matches_step3', 10)

,pat_id,head_id,phrase_case,adp,inf_verb,phrase_nr
0,1427,3,all,,,1
1,1427,3,all,,,1
2,1427,3,all,,,1
3,1427,3,all,,,1
4,1427,3,all,,,1
5,1427,3,all,,,1
6,1427,3,all,,,1
7,1427,3,all,,,1
8,1427,3,all,,,1
9,1427,3,all,,,1


In [7]:
display_sqlite_as_dataframe(RESULT_DB, 'patterns_meta', 10)

,pat_id,phrase_count
0,245,207336
1,2013,118329
2,1625,97861
3,1420,88232
4,478,85378
5,1252,79234
6,1419,72495
7,2361,66183
8,1139,65562
9,2018,61549


In [8]:
display_sqlite_as_dataframe(RESULT_DB, 'verb_phrase_matches', 10)

,pat_id,head_id,phrase_nr
0,1427,3,1
1,794,6,1
2,2013,10,1
3,245,17,1
4,45,23,1
5,245,28,1
6,1938,31,1
7,1139,48,1
8,1934,49,1
9,2013,53,1


In [9]:
display_sqlite_as_dataframe(RESULT_DB, 'verb_matches', 10)

,pat_id,head_id
0,1,52151
1,1,75021
2,1,533958
3,1,603867
4,1,663669
5,1,750763
6,1,1024531
7,1,1112876
8,1,1218427
9,1,1479557
